In [ ]:
# === Environment setup (Kaggle / Colab / Local) ===
import sys, os, glob
from pathlib import Path
import tensorflow as tf

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB  = "google.colab" in sys.modules
if not ON_COLAB:
    try:
        import google.colab
        ON_COLAB = True
    except ImportError:
        pass

if ON_KAGGLE:
    OUT_DIR = Path("/kaggle/working")
    ENV     = "Kaggle"
elif ON_COLAB:
    OUT_DIR = Path("/content")
    ENV     = "Colab"
else:
    OUT_DIR = Path(".")
    ENV     = "Local"

print(f"Environment : {ENV}")
print(f"Output dir  : {OUT_DIR}")
print(f"TF version  : {tf.__version__}")
print(f"GPUs        : {tf.config.list_physical_devices('GPU')}")

In [ ]:
import re
import pandas as pd

KNOWN_EMOTIONS = {"Angry", "Disgusted", "Fearful", "Happy", "Neutral", "Sad"}

def find_emotions_dir(root):
    for c in glob.glob(os.path.join(str(root), "**", ""), recursive=True):
        try:
            subs = {d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))}
        except OSError:
            continue
        if len(KNOWN_EMOTIONS & subs) >= 3:
            return Path(c)
    return None

CANDIDATES = [
    Path("/kaggle/input"),
    Path("/content/Emotions"),
    Path("Emotions"),
    Path("../Emotions"),
    Path.home() / "Downloads" / "Emotions",
]
EMO_DIR = None
for candidate in CANDIDATES:
    if candidate.exists():
        found = find_emotions_dir(candidate)
        if found:
            EMO_DIR = found
            break

if EMO_DIR is None and ON_COLAB:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("uldisvalainis/audio-emotions")
    EMO_DIR = find_emotions_dir(DATA_ROOT)

assert EMO_DIR is not None, (
    "Emotions folder not found.\n"
    "Kaggle: attach uldisvalainis/audio-emotions via Add Input.\n"
    "Colab : upload Emotions/ to /content/ or set up kaggle.json."
)
print("dataset:", EMO_DIR.resolve())

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rows.append({"filename": wav.name, "emotion": emo_path.name,
                     "source": source_of(wav.name), "path": str(wav)})
df = pd.DataFrame(rows)
df = df[df["source"].isin(["CREMA-D", "RAVDESS"])]
df = df[df["emotion"] != "Suprised"].reset_index(drop=True)
print(f"{len(df)} clips, {df['emotion'].nunique()} emotions")
df["emotion"].value_counts()

In [ ]:
import librosa
from tqdm.auto import tqdm

# Header-read only (fast). Keep clips 1–3.5 s; longer ones get truncated to 3.5 s during feature
# extraction anyway, so dropping them here keeps the duration distribution clean.
df["duration"] = [librosa.get_duration(path=p) for p in tqdm(df["path"], desc="duration")]

before = len(df)
df = df[(df["duration"] >= 1.0) & (df["duration"] <= 3.5)].reset_index(drop=True)
print(f"removed {before - len(df)} clips  ({before} -> {len(df)} remain)")
df["emotion"].value_counts()

In [ ]:
# === Utterance-level train/test split (random, all speakers in both sets) ===
from sklearn.model_selection import train_test_split

def speaker_of(filename):
    if re.match(r'^\d+-\d+-', filename):
        return f"RAVDESS-{filename.split('.')[0].split('-')[-1]}"
    if re.match(r'^\d{4}_', filename):
        return f"CREMA-{filename[:4]}"
    return filename

df = df.copy()
df["speaker"] = df["filename"].apply(speaker_of)

tr_idx, te_idx = train_test_split(df.index, test_size=0.2, random_state=42,
                                   stratify=df["emotion"])
dftrain = df.loc[tr_idx].reset_index(drop=True)
dftest  = df.loc[te_idx].reset_index(drop=True)

print(f"train : {len(dftrain)} clips  ({dftrain['speaker'].nunique()} speakers)")
print(f"test  : {len(dftest)} clips  ({dftest['speaker'].nunique()} speakers)")
overlap = set(dftrain["speaker"]) & set(dftest["speaker"])
print(f"shared speakers : {len(overlap)}  (expected > 0 with random split)")
print("\ntrain emotion distribution:")
print(dftrain["emotion"].value_counts())

In [ ]:
import numpy as np
import multiprocessing
from joblib import Parallel, delayed

SR         = 16000
N_MFCC     = 40
HOP        = 512
MAX_FRAMES = int(np.ceil(3.5 * SR / HOP))   # 110 frames — 3.5s cap at 16 kHz
AUG_ROUNDS = 1
PV_WIN     = 7

def _fit2d(m):
    if m.shape[1] < MAX_FRAMES:
        return np.pad(m, ((0, 0), (0, MAX_FRAMES - m.shape[1])))
    return m[:, :MAX_FRAMES]

def _fit1d(a):
    a = a[:MAX_FRAMES] if len(a) >= MAX_FRAMES else np.pad(a, (0, MAX_FRAMES - len(a)))
    return np.tile(a, (N_MFCC, 1))

def _local_pitch_var(f0_nan, win=PV_WIN):
    half = win // 2
    out  = np.zeros(len(f0_nan), dtype="float32")
    for i in range(len(f0_nan)):
        seg = f0_nan[max(0, i - half): i + half + 1]
        seg = seg[~np.isnan(seg)]
        if len(seg) >= 2:
            out[i] = seg.std()
    return out

def features(path, augment=False):
    y, sr = librosa.load(path, sr=SR)
    if augment:
        if np.random.rand() < 0.5:
            y = y + 0.01 * np.random.randn(len(y))
        if np.random.rand() < 0.5:
            y = librosa.effects.pitch_shift(y, sr=sr, n_steps=np.random.uniform(-2, 2))
        if np.random.rand() < 0.5:
            y = librosa.effects.time_stretch(y, rate=np.random.uniform(0.9, 1.1))

    mfcc     = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, hop_length=HOP)
    d1       = librosa.feature.delta(mfcc)
    d2       = librosa.feature.delta(mfcc, order=2)
    rms      = librosa.feature.rms(y=y, hop_length=HOP)[0]
    cen      = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP)[0]
    S        = np.abs(librosa.stft(y, hop_length=HOP))
    flux     = np.concatenate([[0.0], np.sqrt((np.diff(S, axis=1) ** 2).sum(0))])
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=HOP).mean(0)

    # yin (~8x faster than pyin — no probabilistic viterbi decode)
    f0_raw    = librosa.yin(y, fmin=65, fmax=2093, sr=sr, hop_length=HOP).astype(float)
    voiced    = (f0_raw > 65) & (f0_raw < 2093)
    f0_nan    = f0_raw.copy(); f0_nan[~voiced] = np.nan
    f0_frame  = np.where(voiced, f0_raw, 0.0).astype("float32")
    pitch_var = _local_pitch_var(f0_nan)

    v = f0_raw[voiced]
    if len(v) >= 2:
        idx      = np.flatnonzero(voiced).astype(float)
        f0_std   = float(v.std())
        f0_range = float(np.percentile(v, 95) - np.percentile(v, 5))
        slope    = float(np.polyfit(idx, v, 1)[0])
        T        = 1.0 / v
        jitter   = float(np.abs(np.diff(T)).mean() / (T.mean() + 1e-9))
    else:
        f0_std = f0_range = slope = jitter = 0.0

    m       = min(len(rms), len(voiced))
    amp     = rms[:m][voiced[:m]]
    shimmer = float(np.abs(np.diff(amp)).mean() / (amp.mean() + 1e-9)) if len(amp) >= 2 else 0.0
    yh, yp  = librosa.effects.hpss(y)
    hnr     = float(10 * np.log10((yh ** 2).sum() / ((yp ** 2).sum() + 1e-9) + 1e-9))

    tensor = np.stack([
        _fit2d(mfcc), _fit2d(d1), _fit2d(d2),
        _fit1d(rms), _fit1d(f0_frame), _fit1d(cen),
        _fit1d(flux), _fit1d(contrast),
        _fit1d(pitch_var),
    ], axis=-1).astype("float32")                        # (40, MAX_FRAMES, 9)

    scalars = np.array([f0_std, f0_range, slope, jitter, shimmer, hnr], dtype="float32")
    return tensor, scalars

def build_X(frame, desc, augment=False):
    paths = list(frame["path"])
    results = [features(p, augment) for p in tqdm(paths, desc=desc)]
    T  = np.stack([r[0] for r in results]).astype("float32")
    Sc = np.stack([r[1] for r in results]).astype("float32")
    return T, Sc

CACHE = OUT_DIR / "myfeatures_cache.npz"
if CACHE.exists():
    dat     = np.load(str(CACHE))
    X_train, S_train = dat["X_train"], dat["S_train"]
    X_test,  S_test  = dat["X_test"],  dat["S_test"]
    print("loaded from cache:", X_train.shape)
else:
    X_train, S_train = build_X(dftrain, "train")
    for r in range(AUG_ROUNDS):
        Xa, Sa  = build_X(dftrain, f"train-aug{r+1}", augment=True)
        X_train = np.concatenate([X_train, Xa])
        S_train = np.concatenate([S_train, Sa])
    X_test, S_test = build_X(dftest, "test")
    np.savez_compressed(str(CACHE),
                        X_train=X_train, S_train=S_train,
                        X_test=X_test,   S_test=S_test)
    print("extracted + cached:", X_train.shape)

print("X_train:", X_train.shape, " S_train:", S_train.shape,
      " X_test:",  X_test.shape,  " S_test:",  S_test.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
ytr_int = le.fit_transform(dftrain["emotion"])
yte_int = le.transform(dftest["emotion"])
print("classes:", list(le.classes_))

# Class weights from the ORIGINAL train labels (handles Neutral/Disgusted being under-represented)
cw = compute_class_weight("balanced", classes=np.unique(ytr_int), y=ytr_int)
class_weight = dict(enumerate(cw))
print("class_weight:", {le.classes_[k]: round(v, 3) for k, v in class_weight.items()})

# Repeat labels to match the (1 clean + AUG_ROUNDS) copies concatenated in build_X
ytr = to_categorical(np.tile(ytr_int, 1 + AUG_ROUNDS))
yte = to_categorical(yte_int)

# --- CNN tensor: per-channel standardization using TRAIN stats only ---
axes = (0, 1, 2)                                   # over samples, mfcc rows, time -> one stat per channel
mean = X_train.mean(axis=axes, keepdims=True)
std  = X_train.std(axis=axes, keepdims=True)
X_train_n = (X_train - mean) / (std + 1e-9)
X_test_n  = (X_test  - mean) / (std + 1e-9)

# --- scalar branch: per-feature standardization using TRAIN stats only ---
s_mean = S_train.mean(axis=0, keepdims=True)
s_std  = S_train.std(axis=0, keepdims=True)
S_train_n = (S_train - s_mean) / (s_std + 1e-9)
S_test_n  = (S_test  - s_mean) / (s_std + 1e-9)

print("X_train_n:", X_train_n.shape, " S_train_n:", S_train_n.shape,
      " X_test_n:", X_test_n.shape, " S_test_n:", S_test_n.shape)


In [ ]:
# === Feature diagnostics — find which channel has NaN / inf ===
TENSOR_CHANNELS = ["mfcc", "delta1", "delta2", "rms", "f0", "centroid", "flux", "contrast", "pitch_var"]
SCALAR_NAMES    = ["f0_std", "f0_range", "slope", "jitter", "shimmer", "hnr"]

print("── Tensor channels (X_train_n) ──")
for i, name in enumerate(TENSOR_CHANNELS):
    ch = X_train_n[..., i]
    has_nan = np.isnan(ch).any()
    has_inf = np.isinf(ch).any()
    print(f"  [{i}] {name:12s}  min={ch.min():.3f}  max={ch.max():.3f}  nan={has_nan}  inf={has_inf}")

print("\n── Scalar features (S_train_n) ──")
for i, name in enumerate(SCALAR_NAMES):
    col = S_train_n[:, i]
    has_nan = np.isnan(col).any()
    has_inf = np.isinf(col).any()
    print(f"  [{i}] {name:10s}  min={col.min():.3f}  max={col.max():.3f}  nan={has_nan}  inf={has_inf}")

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# float32 — mixed precision is not worth the instability on a 550K model
tf.keras.mixed_precision.set_global_policy("float32")
print("compute dtype:", tf.keras.mixed_precision.global_policy().compute_dtype)

assert not np.isnan(X_train_n).any(), "NaN in X_train_n"
assert not np.isnan(S_train_n).any(), "NaN in S_train_n"
print("Input check passed — no NaN")

L2 = regularizers.l2(1e-4)

try:
    _register = keras.saving.register_keras_serializable
except AttributeError:
    _register = keras.utils.register_keras_serializable

@_register()
class ReduceSum(keras.layers.Layer):
    def call(self, x):
        return tf.reduce_sum(x, axis=1)

# CNN branch — 9 time-varying channels
img_in = keras.Input(shape=X_train_n.shape[1:], name="spectral")
x = layers.GaussianNoise(0.1)(img_in)
for f in (32, 64, 128):
    x = layers.Conv2D(f, 3, padding="same", activation="relu", kernel_regularizer=L2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.SpatialDropout2D(0.1)(x)
x = layers.Permute((2, 1, 3))(x)
tdim, fdim, cdim = x.shape[1], x.shape[2], x.shape[3]
x = layers.Reshape((tdim, fdim * cdim))(x)

# BiGRU + attention pooling over time
x       = layers.Bidirectional(layers.GRU(96, return_sequences=True, kernel_regularizer=L2))(x)
score   = layers.Dense(1, activation="tanh")(x)
weights = layers.Softmax(axis=1)(score)
x       = layers.Multiply()([x, weights])
seq     = ReduceSum()(x)

# Dense branch — 6 clip-level scalars
sc_in = keras.Input(shape=(S_train_n.shape[1],), name="scalars")
s = layers.Dense(32, activation="relu", kernel_regularizer=L2)(sc_in)
s = layers.Dropout(0.3)(s)

# Merge + classify
h   = layers.Concatenate()([seq, s])
h   = layers.Dropout(0.5)(h)
h   = layers.Dense(128, activation="relu", kernel_regularizer=L2)(h)
h   = layers.Dropout(0.5)(h)
out = layers.Dense(ytr.shape[1], activation="softmax")(h)

model = keras.Model([img_in, sc_in], out)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)
model.summary()

CKPT_PATH = str(OUT_DIR / "mypick_best.keras")
history = model.fit(
    {"spectral": X_train_n, "scalars": S_train_n}, ytr,
    validation_data=({"spectral": X_test_n, "scalars": S_test_n}, yte),
    epochs=80, batch_size=256,
    class_weight=class_weight,
    callbacks=[
        keras.callbacks.ModelCheckpoint(CKPT_PATH, monitor="val_accuracy",
                                        save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=4, min_lr=1e-5),
    ],
)
print(f"best model saved -> {CKPT_PATH}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model = keras.models.load_model(
    CKPT_PATH,
    custom_objects={"ReduceSum": ReduceSum},
    safe_mode=False,
)

pred = model.predict({"spectral": X_test_n, "scalars": S_test_n}).argmax(1)
true = yte.argmax(1)
print(classification_report(true, pred, target_names=le.classes_))

cm = confusion_matrix(true, pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("CNN+BiGRU+attn | 16 kHz, 3.5s cap")
plt.show()

stats_path = str(OUT_DIR / "mypick_norm_stats.npz")
np.savez(stats_path, mean=mean, std=std, s_mean=s_mean, s_std=s_std,
         classes=le.classes_, sr=SR, max_frames=MAX_FRAMES)
print(f"model  -> {CKPT_PATH}")
print(f"stats  -> {stats_path}")

if ON_COLAB:
    from google.colab import files
    files.download(CKPT_PATH)
    files.download(stats_path)